# 05 — Designing CPs by Composing Operators

The interesting structure of an SRG is determined entirely by the *input* tiling. By pre-composing Conway operators on top of a regular tiling you can quickly explore a wide design space.

This notebook is a **recipe book**: each cell shows one composition, with a one-line description.

In [ ]:
import matplotlib
matplotlib.rcParams['figure.figsize'] = (5, 5)
import matplotlib.pyplot as plt
import numpy as np

import eucare as ec
from eucare import (
    conway,
    example_graphs,
    example_tilesets,
    overlap,
    plotting,
    reciprocal_figures,
    rendering,
)


def plot_g(G, ax=None, color='black', linewidth=1.0):
    """Draw the edges of a half-edge graph G on `ax` (or the current axes)."""
    if ax is None:
        ax = plt.gca()
    lines = np.array([
        [G.geometry.to_euclidean(h.orig['pos']),
         G.geometry.to_euclidean(h.dest['pos'])]
        for h in G.halfedges_representing_edges()
    ])
    plotting.plot_lines(lines, ax=ax, colors=color, linewidths=linewidth)
    plotting.set_equal_aspect(ax)
    ax.axis('off')


In [ ]:
from eucare.search_trees import face_bfs_tree
from eucare.reciprocal_figures import assign_this_way_by_face_z_order, make_SRG


def srg_pipeline(G):
    """Run the standard SRG pipeline: BFS z-order -> SRG -> recompute."""
    central = min(G.faces, key=lambda f: np.linalg.norm(f.midpoint()))
    central['z_order'] = 0
    for orig, dest in face_bfs_tree(central):
        dest['z_order'] = orig['z_order'] + 1
    assign_this_way_by_face_z_order(G)
    SRG = make_SRG(G)
    SRG.recompute_lengths_and_angles()
    return SRG


In [ ]:
def show_pair(G, title, axes):
    plot_g(G, ax=axes[0]); axes[0].set_title(f'tiling: {title}')
    plot_g(srg_pipeline(G), ax=axes[1]); axes[1].set_title('SRG')


## Recipe 1: ambo on hexagons

In [ ]:
G = example_graphs.from_tiles(example_tilesets.platonic(6), rings=2)
G = conway.ambo_graph()(G, delete_on_border=True)
G.recompute_lengths_and_angles()
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
show_pair(G, 'ambo(hex)', axes); plt.tight_layout(); plt.show()


## Recipe 2: kis on the 3.3.4.3.4 Archimedean tiling

In [ ]:
G = example_graphs.from_tiles(example_tilesets.t_3_3_4_3_4(), rings=2)
G = conway.kis_graph()(G, delete_on_border=True)
G.recompute_lengths_and_angles()
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
show_pair(G, 'kis(3.3.4.3.4)', axes); plt.tight_layout(); plt.show()


## Recipe 3: alternating-flagstone on a square tiling

In [ ]:
G = example_graphs.from_tiles(example_tilesets.platonic(4), rings=2)
G = conway.alternating_flagstone_graph()(G, delete_on_border=True)
G.recompute_lengths_and_angles()
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
show_pair(G, 'alt-flagstone(squares)', axes); plt.tight_layout(); plt.show()


## Recipe 4: Pietro-Vitelli flagstones on hexagons

In [ ]:
G = example_graphs.from_tiles(example_tilesets.platonic(6), rings=2)
G = conway.flagstone_pvitelli_graph()(G, delete_on_border=True)
G.recompute_lengths_and_angles()
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
show_pair(G, 'flagstone_pvitelli(hex)', axes); plt.tight_layout(); plt.show()


## Going further

- Conway operators compose freely: e.g. `kis(ambo(platonic(6)))`.
- All Archimedean tilings in `example_tilesets` are valid starting points; same for `curved_platonic` (see `02_Curved_Geometries`).
- The legacy notebooks `Intersecting Cylinders.ipynb` and `Winni Leung Corrugations.ipynb` contain more advanced compositions.